In [1]:
import pandas as pd
from google.cloud import storage
import glob
import os
from datetime import datetime
import hashlib


# List all files in the bucket/folder that match the pattern
def list_gcs_files(bucket_name, prefix, suffix):
    storage_client = storage.Client()
    bucket = storage_client.get_bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)
    return [f"gs://{bucket_name}/{blob.name}" for blob in blobs if blob.name.endswith(suffix)]

# Get all position parquet files
bucket_name = "translinkdata"
prefix = "raw_data/"
suffix = "position.parquet"

# parquet_files = list_gcs_files(bucket_name, prefix, suffix)


In [2]:
def generate_file_id(filename):
    """Generate a unique ID for a file using SHA-256 hash."""
    return hashlib.sha256(os.path.basename(filename).encode()).hexdigest()[:16]

In [3]:
def read_file_names(data_path, pattern):
    # data_path = "../data/lambda_sync/raw_data"
    # pattern = "realtime"
    # Create full pattern and print it for debugging
    glob_pattern = os.path.join(data_path, f"*_{pattern}.parquet")
    print(f"Searching with pattern: {glob_pattern}")

    # List all files in directory for debugging
    # print(f"Files in directory: {os.listdir(data_path)}")

    # Find all parquet files matching the pattern
    matching_files = glob.glob(glob_pattern)
    # print(f"Found files: {matching_files}")

    if not matching_files:
        raise ValueError(f"No parquet files found matching pattern '{pattern}'")

    # Read and combine all matching parquet files
    
#     batch_size = 400

#     print(len(matching_files)/batch_size)

#     buckets = []
#     bucket_stats = {}

#     for i in range(0, len(matching_files), batch_size):
#         buckets.append(matching_files[i:i+batch_size])
#         # bucket_stats[i] = len(matching_files[i:i+batch_size])

    

    # print(bucket_stats)
    # return buckets
    
    print(len(matching_files))
    return matching_files

In [4]:
position_parquet_list  = read_file_names("../data/lambda_sync/raw_data", 'position')
realtime_parquet_list  = read_file_names("../data/lambda_sync/raw_data", 'realtime')
weather_parquet_list = glob.glob("../data/lambda_sync/weather_data/*.parquet")

Searching with pattern: ../data/lambda_sync/raw_data/*_position.parquet
12548
Searching with pattern: ../data/lambda_sync/raw_data/*_realtime.parquet
12683


In [5]:
# Read all parquet files into a single dataframe
pd.read_parquet(position_parquet_list[0:1]).info()
pd.read_parquet(realtime_parquet_list[0:1]).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568 entries, 0 to 567
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     568 non-null    object        
 1   trip_id                568 non-null    object        
 2   start_date             568 non-null    object        
 3   schedule_relationship  568 non-null    int64         
 4   route_id               568 non-null    object        
 5   direction_id           568 non-null    int64         
 6   vehicle_id             568 non-null    object        
 7   vehicle_label          568 non-null    object        
 8   latitude               568 non-null    float64       
 9   longitude              568 non-null    float64       
 10  current_stop_sequence  568 non-null    int64         
 11  current_status         568 non-null    int64         
 12  timestamp              568 non-null    int64         
 13  stop_

In [6]:
def define_weather_schema(parquet_list):
    dtype_schema = {
        'code':'object',
        'updateTime':'object',
        'fxLink':'object',
        'now.obsTime':'object',
        'now.temp':'int32',
        'now.feelsLike':'int32',
        'now.icon':'int32',
        'now.text':'object',
        'now.wind360':'int32',
        'now.windDir':'object',
        'now.windScale':'int32',
        'now.windSpeed':'int32',
        'now.humidity':'int32',
        'now.precip':'float64',
        'now.pressure':'int32',
        'now.vis':'int32',
        'now.cloud':'int32',
        'now.dew':'int32',
        'refer.sources':'object',
        'refer.license':'object',
    }

    df_list = []

    for f in parquet_list:
        temp_df = pd.read_parquet(f).astype(dtype_schema)
        temp_df['scrape_id'] = generate_file_id(f)
        df_list.append(temp_df)
        del temp_df

    df = pd.concat(df_list)

    # First localize the naive timestamp to UTC since we know it's in UTC
    df['current_datetime'] = pd.to_datetime(df['now.obsTime']).dt.tz_localize('UTC')
    df['current_datetime_unix'] = df['current_datetime'].map(datetime.timestamp).astype(int)

    df['year'] = df['current_datetime'].dt.year
    df['month'] = df['current_datetime'].dt.month
    df['day'] = df['current_datetime'].dt.day
    df['current_date'] = df['current_datetime'].dt.date
    df['current_datetime_str'] = df['current_datetime'].dt.tz_convert('America/Vancouver').astype(str)
    
    df = df.drop(columns = ['current_datetime'])

    print(df.info())
    return df

    

In [17]:

    
    
def define_realtime_schema(parquet_list):
    dtype_schema = {
        'id': 'object',
        'is_deleted': 'bool',
        'trip_id': 'object',
        'start_date': 'object',
        'schedule_relationship': 'int64',
        'route_id': 'object',
        'direction_id': 'int64',
        'vehicle_id': 'object',
        'vehicle_label': 'object',
        'stop_sequence': 'int64',
        'stop_id': 'object',
        'arrival_delay': 'float64',
        'arrival_time': 'float64',
        'departure_delay': 'float64',
        'departure_time': 'float64',
        'stop_schedule_relationship': 'int64'
    }
    
    # Read and concatenate with enforced dtypes
    df_list = []

    for f in parquet_list:
        temp_df = pd.read_parquet(f).astype(dtype_schema)
        temp_df['scrape_id'] = generate_file_id(f)
        # temp_df['current_datetime'] = pd.to_datetime(temp_df['current_datetime'], unit='s',utc=True)
        temp_df['current_datetime_unix'] = temp_df['current_datetime'].map(datetime.timestamp).astype(int)
        
        temp_df['year'] = temp_df['current_datetime'].dt.year
        temp_df['month'] = temp_df['current_datetime'].dt.month
        temp_df['day'] = temp_df['current_datetime'].dt.day
        temp_df['current_date']  = temp_df['current_datetime'].dt.date
        # Drop the datetime column

        temp_df['current_datetime_str'] = temp_df['current_datetime'].astype(str)
        temp_df = temp_df.drop(columns=['current_datetime'])
        df_list.append(temp_df)
        del temp_df

    df = pd.concat(df_list)
    
    # Convert current_datetime to unix timestamp and rename

    print(df.info())
    return df
    

In [21]:
def define_position_schema(parquet_list):
    
    dtype_schema = {
    'id': 'object',
    'trip_id': 'object',
    'start_date': 'object',
    'schedule_relationship': 'int64',
    'route_id': 'object',
    'direction_id': 'int64',
    'vehicle_id': 'object',
    'vehicle_label': 'object',
    'latitude': 'float64',
    'longitude': 'float64',
    'current_stop_sequence': 'int64',
    'current_status': 'int64',
    'timestamp': 'int64',
    'stop_id': 'object'
    }

    # Read and concatenate with enforced dtypes
    df_list = []

    for f in parquet_list:
        temp_df = pd.read_parquet(f).astype(dtype_schema)
        temp_df['scrape_id'] = generate_file_id(f)
        df_list.append(temp_df)
        del temp_df

    df = pd.concat(df_list)
    
    # df['current_datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    # df['current_datetime'] = pd.to_datetime(df['current_datetime'], unit='s')
    df['current_datetime'] = df['current_datetime'].dt.tz_localize('UTC')
    df['current_datetime_unix'] = df['current_datetime'].map(datetime.timestamp).astype(int)
    
    df['year'] = df['current_datetime'].dt.year
    df['month'] = df['current_datetime'].dt.month
    df['day'] = df['current_datetime'].dt.day
    df['current_date']  = df['current_datetime'].dt.date
    
    df['current_datetime_str'] = df['current_datetime'].dt.tz_convert('America/Vancouver',).astype(str)
    df['current_datetime_str_utc'] = df['current_datetime'].astype(str)
    df['current_datetime_converted_utc'] = pd.to_datetime(df['current_datetime_unix'],unit = 's').dt.tz_localize('UTC')
    df['current_datetime_converted_van'] = df['current_datetime_converted_utc'].dt.tz_convert('America/Vancouver',)
    df = df.drop(columns = ['current_datetime'])
    
    print(df.info())
    return df 

In [22]:
define_position_schema(position_parquet_list[0:1])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568 entries, 0 to 567
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype                            
---  ------                          --------------  -----                            
 0   id                              568 non-null    object                           
 1   trip_id                         568 non-null    object                           
 2   start_date                      568 non-null    object                           
 3   schedule_relationship           568 non-null    int64                            
 4   route_id                        568 non-null    object                           
 5   direction_id                    568 non-null    int64                            
 6   vehicle_id                      568 non-null    object                           
 7   vehicle_label                   568 non-null    object                           
 8   latitude            

,id,trip_id,start_date,schedule_relationship,route_id,direction_id,vehicle_id,vehicle_label,latitude,longitude,...,scrape_id,current_datetime_unix,year,month,day,current_date,current_datetime_str,current_datetime_str_utc,current_datetime_converted_utc,current_datetime_converted_van
0,14121792,14121792,20250104,0,30055,0,19524,19524,49.286499,-123.140701,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.859985-08:00,2025-01-04 16:34:48.859985+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
1,14010917,14010917,20250104,0,6637,1,18313,18313,49.281483,-123.102264,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.859998-08:00,2025-01-04 16:34:48.859998+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
2,13998796,13998796,20250104,0,6614,0,2277,2277,49.283852,-123.109154,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.860005-08:00,2025-01-04 16:34:48.860005+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
3,14000152,14000152,20250104,0,6616,1,2220,2220,49.284184,-123.137070,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.860012-08:00,2025-01-04 16:34:48.860012+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
4,13997768,13997768,20250104,0,6612,0,21023,21023,49.272484,-123.145050,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.860019-08:00,2025-01-04 16:34:48.860019+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,14123939,14123939,20250104,0,11696,0,19516,19516,49.049198,-123.098267,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893725-08:00,2025-01-04 16:34:48.893725+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
564,14123966,14123966,20250104,0,11696,1,19513,19513,49.047482,-123.105637,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893731-08:00,2025-01-04 16:34:48.893731+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
565,14003778,14003778,20250104,0,6622,1,2239,2239,49.260868,-123.139702,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893738-08:00,2025-01-04 16:34:48.893738+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00
566,14003836,14003836,20250104,0,6622,1,2113,2113,49.281216,-123.051468,...,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893744-08:00,2025-01-04 16:34:48.893744+00:00,2025-01-04 16:34:48+00:00,2025-01-04 08:34:48-08:00


In [8]:
def write_to_gcs(df, path):
    return df.to_parquet( path,  partition_cols=['year', 'month', 'day'],
            engine='pyarrow',
            index=False)

In [9]:
# weather_parquet_with_schema = define_weather_schema(weather_parquet_list)
# write_to_gcs( df = weather_parquet_with_schema, path = '/Users/JLY/coding_projects/projects/translink_bus/ML/data/spark_partitioned/weather')
# del weather_parquet_with_schema

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568 entries, 0 to 567
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        568 non-null    object 
 1   trip_id                   568 non-null    object 
 2   start_date                568 non-null    object 
 3   schedule_relationship     568 non-null    int64  
 4   route_id                  568 non-null    object 
 5   direction_id              568 non-null    int64  
 6   vehicle_id                568 non-null    object 
 7   vehicle_label             568 non-null    object 
 8   latitude                  568 non-null    float64
 9   longitude                 568 non-null    float64
 10  current_stop_sequence     568 non-null    int64  
 11  current_status            568 non-null    int64  
 12  timestamp                 568 non-null    int64  
 13  stop_id                   568 non-null    object 
 14  scrape_id 

,id,trip_id,start_date,schedule_relationship,route_id,direction_id,vehicle_id,vehicle_label,latitude,longitude,...,timestamp,stop_id,scrape_id,current_datetime_unix,year,month,day,current_date,current_datetime_str,current_datetime_str_utc
0,14121792,14121792,20250104,0,30055,0,19524,19524,49.286499,-123.140701,...,1736008459,1,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.859985-08:00,2025-01-04 16:34:48.859985+00:00
1,14010917,14010917,20250104,0,6637,1,18313,18313,49.281483,-123.102264,...,1736008455,22,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.859998-08:00,2025-01-04 16:34:48.859998+00:00
2,13998796,13998796,20250104,0,6614,0,2277,2277,49.283852,-123.109154,...,1736008444,36,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.860005-08:00,2025-01-04 16:34:48.860005+00:00
3,14000152,14000152,20250104,0,6616,1,2220,2220,49.284184,-123.137070,...,1736008405,52,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.860012-08:00,2025-01-04 16:34:48.860012+00:00
4,13997768,13997768,20250104,0,6612,0,21023,21023,49.272484,-123.145050,...,1736008411,72,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.860019-08:00,2025-01-04 16:34:48.860019+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,14123939,14123939,20250104,0,11696,0,19516,19516,49.049198,-123.098267,...,1736008399,12992,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893725-08:00,2025-01-04 16:34:48.893725+00:00
564,14123966,14123966,20250104,0,11696,1,19513,19513,49.047482,-123.105637,...,1736008417,12995,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893731-08:00,2025-01-04 16:34:48.893731+00:00
565,14003778,14003778,20250104,0,6622,1,2239,2239,49.260868,-123.139702,...,1736008404,13004,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893738-08:00,2025-01-04 16:34:48.893738+00:00
566,14003836,14003836,20250104,0,6622,1,2113,2113,49.281216,-123.051468,...,1736008444,13011,29280f402559f229,1736008488,2025,1,4,2025-01-04,2025-01-04 08:34:48.893744-08:00,2025-01-04 16:34:48.893744+00:00


In [10]:

print(pd.read_parquet(position_parquet_list[0:1]).info())
print(pd.read_parquet(weather_parquet_list[0:1]).info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568 entries, 0 to 567
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     568 non-null    object        
 1   trip_id                568 non-null    object        
 2   start_date             568 non-null    object        
 3   schedule_relationship  568 non-null    int64         
 4   route_id               568 non-null    object        
 5   direction_id           568 non-null    int64         
 6   vehicle_id             568 non-null    object        
 7   vehicle_label          568 non-null    object        
 8   latitude               568 non-null    float64       
 9   longitude              568 non-null    float64       
 10  current_stop_sequence  568 non-null    int64         
 11  current_status         568 non-null    int64         
 12  timestamp              568 non-null    int64         
 13  stop_

In [ ]:
# position_parquet_with_schema = define_position_schema(position_parquet_list)
# write_to_gcs(df = position_parquet_with_schema, path = '/Users/JLY/coding_projects/projects/translink_bus/ML/data/spark_partitioned/position')
# del position_parquet_with_schema

<class 'pandas.core.frame.DataFrame'>
Index: 7649545 entries, 0 to 778
Data columns (total 22 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   id                        object 
 1   trip_id                   object 
 2   start_date                object 
 3   schedule_relationship     int64  
 4   route_id                  object 
 5   direction_id              int64  
 6   vehicle_id                object 
 7   vehicle_label             object 
 8   latitude                  float64
 9   longitude                 float64
 10  current_stop_sequence     int64  
 11  current_status            int64  
 12  timestamp                 int64  
 13  stop_id                   object 
 14  scrape_id                 object 
 15  current_datetime_unix     int64  
 16  year                      int32  
 17  month                     int32  
 18  day                       int32  
 19  current_date              object 
 20  current_datetime_str      object 

In [12]:
# define_realtime_schema(realtime_parquet_list[0:1])

In [13]:
# realtime_parquet_with_schema = define_realtime_schema(realtime_parquet_list)
# write_to_gcs(df =realtime_parquet_with_schema, path = '/Users/JLY/coding_projects/projects/translink_bus/ML/data/spark_partitioned/realtime')
# del realtime_parquet_with_schema